In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("../data/merged/merged_data.csv", low_memory=False)

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 25429 entries, 0 to 25428
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   NITRATE(PPM)   25429 non-null  float64
 1   PH             25429 non-null  float64
 2   AMMONIA(mg/l)  25429 non-null  float64
 3   TEMP           25429 non-null  float64
 4   DO             25429 non-null  float64
 5   TURBIDITY      25429 non-null  float64
 6   datetime       25429 non-null  str    
 7   feed_window    25429 non-null  int64  
dtypes: float64(6), int64(1), str(1)
memory usage: 1.6 MB


In [5]:
df['datetime'] = pd.to_datetime(df['datetime'])
df.sort_values(by='datetime').reset_index(drop=True)
df.dtypes

NITRATE(PPM)            float64
PH                      float64
AMMONIA(mg/l)           float64
TEMP                    float64
DO                      float64
TURBIDITY               float64
datetime         datetime64[us]
feed_window               int64
dtype: object

## Dissolved Oxygen Stress Index (DO_stress_idx)
This index measures how far the pond has fallen below the 5.0 threshold.<br>
5.0 mg/L is the critical threshold for fish metabolism.<br> 
Below this, fish cannot efficiently break down food; they physically cannot eat even if they want to. Above it, there is no DO stress regardless of whether DO is 6 or 20.

In [8]:
DO_THRESHOLD = 5.0
# max(0, 5.0 - DO)
df['DO_stress_idx'] = np.maximum(0, DO_THRESHOLD - df['DO'])
print(df[['datetime', 'DO', 'DO_stress_idx']].head().to_string(index=False))
print(f"\nMean Per Class {df.groupby('feed_window')['DO_stress_idx'].mean().round(4)}")

           datetime      DO  DO_stress_idx
2022-02-01 00:00:00 8.21866            0.0
2022-02-01 00:20:00 6.23826            0.0
2022-02-01 00:40:00 6.33728            0.0
2022-02-01 02:00:00 7.12944            0.0
2022-02-01 02:40:00 9.40690            0.0

Mean Per Class feed_window
0    0.0190
1    0.0000
2    0.5335
Name: DO_stress_idx, dtype: float64


Class 0 (Prime Feed)  : 0.0190  ← basically zero, pond is fine<br>
Class 1 (Reduce Feed) : 0.0000  ← literally zero, DO is perfectly safe<br>
Class 2 (Halt Feeding): 0.5335  ← above threshold, DO is low<br>
The Reduce Feed class has a DO stress index of exactly zero. <br>That means when the model needs to recommend Reduce Feed, DO is not the reason.

## pH Deviation (ph_deviation)
Fish physiology works optimally at pH 7.0 (neutral). Stress rises in both directions, a pond that is too acidic (pH 5.5) is just as harmful as one that is too alkaline (pH 8.5). So we do not care about the direction, only the distance from 7.0.

In [9]:
PH_OPTIMAL = 7.0

# abs(pH - 7.0)
df['pH_deviation'] = np.abs(df['PH'] - PH_OPTIMAL)
print(df[['datetime', 'PH', 'pH_deviation']].head(6).to_string(index=False))
print(f"\nMean per class:\n{df.groupby('feed_window')['pH_deviation'].mean().round(4)}")

           datetime    PH  pH_deviation
2022-02-01 00:00:00 5.656         1.344
2022-02-01 00:20:00 5.858         1.142
2022-02-01 00:40:00 5.454         1.546
2022-02-01 02:00:00 5.555         1.445
2022-02-01 02:40:00 5.858         1.142
2022-02-01 03:40:00 5.353         1.647

Mean per class:
feed_window
0    0.9523
1    1.0825
2    0.9594
Name: pH_deviation, dtype: float64


## Thermal Stress (thermal_stress)
